# Convolutional Neural Networks (CNN)
<font color='blue'>
<br>Content: 
* [Loading the Data Set](#1)
* [Normalization, Reshape and Label Encoding ](#2)
* [Train Test Split](#3)
* [Convolutional Neural Network](#4)
    * [What is Convolution Operation?](#5)
    * [Same Padding](#6)
    * [Max Pooling](#7)
    * [Flattening](#8)
    * [Full Connection](#9)
* [Implementing with Keras](#10)
    * [Create Model](#11)
    * [Define Optimizer](#12)
    * [Compile Model](#13)
    * [Epochs and Batch Size](#14)
    * [Data Augmentation](#15)
    * [Fit the Model](#16)
    * [Evaluate the Model](#17)
* [Deep Learning Tutorial for Beginners](https://www.kaggle.com/kanncaa1/deep-learning-tutorial-for-beginners)
* [Artificial Neural Network with Pytorch](https://www.kaggle.com/kanncaa1/pytorch-tutorial-for-deep-learning-lovers)
* [Convolutional Neural Network with Pytorch](https://www.kaggle.com/kanncaa1/pytorch-tutorial-for-deep-learning-lovers)
* [Recurrent Neural Network with Pytorch](https://www.kaggle.com/kanncaa1/recurrent-neural-network-with-pytorch)
* [Conclusion](#18)


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras import layers, models
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import os

print(os.listdir("../input/face-recognition-dataset"))
reduce_lr = ReduceLROnPlateau(
    monitor="val_acc",   
    factor=0.1,
    patience=5,
    min_lr=1e-6
)

<a id="1"></a>
## Loading the Data Set
* In this part we load and visualize the data.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

data_dir = "../input/face-recognition-dataset" 
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    data_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    subset="training"
)

val_gen = datagen.flow_from_directory(
    data_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    subset="validation"
)

In [ ]:
X_batch, y_batch = next(train_gen)
print("Batch shape:", X_batch.shape)
print("Labels shape:", y_batch.shape)

In [ ]:
import pandas as pd
class_counts = pd.Series(train_gen.classes).value_counts()
label_map = {v: k for k, v in train_gen.class_indices.items()}
class_counts.index = class_counts.index.map(label_map)
plt.figure(figsize=(15,7))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="icefire")
plt.xticks(rotation=90)
plt.title("Number of face images per class")
plt.xlabel("Person")
plt.ylabel("Count")
plt.show()

In [ ]:
X_batch, y_batch = next(train_gen)  

plt.figure(figsize=(12,6))
for i in range(9):   
    plt.subplot(3,3,i+1)
    plt.imshow(X_batch[i])
    label_index = np.argmax(y_batch[i])   # get class index
    label_name = list(train_gen.class_indices.keys())[label_index]  
    plt.title(label_name)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
X_batch, y_batch = next(train_gen)

plt.figure(figsize=(12, 6))
for i in range(9):  # plot first 9 images
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_batch[i])  # images are already 224x224x3 RGB
    label_index = np.argmax(y_batch[i])  # find which class
    label_name = list(train_gen.class_indices.keys())[label_index]  # map index to class name
    plt.title(label_name)
    plt.axis("off")

plt.tight_layout()
plt.show()

<a id="2"></a>
## Normalization, Reshape and Label Encoding 
* Normalization
    * We perform a grayscale normalization to reduce the effect of illumination's differences.
    * If we perform normalization, CNN works faster.
* Reshape
    * Train and test images (28 x 28) 
    * We reshape all data to 28x28x1 3D matrices.
    * Keras needs an extra dimension in the end which correspond to channels. Our images are gray scaled so it use only one channel. 
* Label Encoding  
    * Encode labels to one hot vectors 
        * 2 => [0,0,1,0,0,0,0,0,0,0]
        * 4 => [0,0,0,0,1,0,0,0,0,0]

In [ ]:
X_batch, y_batch = next(train_gen)
print("Image pixel range:", X_batch.min(), "to", X_batch.max())

In [ ]:
X_batch, y_batch = next(train_gen)
print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)

In [ ]:
train_gen = datagen.flow_from_directory(
    data_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical", 
    subset="training"
)

<a id="3"></a>
## Train Test Split
* We split the data into train and test sets.
* test size is 10%.
* train size is 90%.

In [ ]:
print("Train samples:", train_gen.samples)
print("Validation samples:", val_gen.samples)
print("Number of classes:", train_gen.num_classes)


In [ ]:
X_batch, y_batch = next(train_gen)  # get one batch

plt.figure(figsize=(6,6))
plt.imshow(X_batch[2])  # directly show RGB image
label_index = np.argmax(y_batch[2])  # one-hot → class index
label_name = list(train_gen.class_indices.keys())[label_index]
plt.title(f"Label: {label_name}")
plt.axis("off")
plt.show()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dropout, Flatten, Dense
from tensorflow.keras.optimizers import Adam

num_classes = train_gen.num_classes  # e.g., 31

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPool2D(2,2),
    Dropout(0.25),

    Conv2D(64, (3,3), activation='relu'),
    MaxPool2D(2,2),
    Dropout(0.25),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer=Adam(lr=0.0001), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()


<a id="12"></a>
### Define Optimizer   
* Adam optimizer: Change the learning rate


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

num_classes = train_gen.num_classes  # automatically from ImageDataGenerator

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPool2D(pool_size=(2,2)),
    Dropout(0.25),

    Conv2D(64, (3,3), activation='relu'),
    MaxPool2D(pool_size=(2,2)),
    Dropout(0.25),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

optimizer = Adam(lr=0.001)  # TF1.x/Keras still uses `lr`
model.compile(optimizer=optimizer,
              loss="categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    verbose=1
)

lr_reduction = ReduceLROnPlateau(
    monitor='val_accuracy',
    patience=3,
    verbose=1,
    factor=0.5,
    min_lr=1e-6
)


In [ ]:
def safe_generator(generator):
    while True:
        try:
            yield next(generator)
        except OSError:
            print("Skipped a corrupted image.")
            continue

train_gen_safe = safe_generator(train_gen)
val_gen_safe   = safe_generator(val_gen)

history = model.fit_generator(
    generator=train_gen_safe,
    steps_per_epoch=train_gen.samples // train_gen.batch_size,
    validation_data=val_gen_safe,
    validation_steps=val_gen.samples // val_gen.batch_size,
    epochs=20,
    callbacks=[lr_reduction, early_stop]
)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,              
    validation_split=0.2,        
    rotation_range=5,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
train_gen = datagen.flow_from_directory(
    data_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    subset="training"
)

val_gen = datagen.flow_from_directory(
    data_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    subset="validation"
)


In [ ]:
# Fit the model
history = model.fit_generator(datagen.flow(X_train,Y_train, batch_size=batch_size),
                              epochs = epochs, validation_data = (X_val,Y_val), steps_per_epoch=X_train.shape[0] // batch_size)

In [ ]:
# Plot the loss and accuracy curves for training and validation 
plt.plot(history.history['val_loss'], color='b', label="validation loss")
plt.title("Test Loss")
plt.xlabel("Number of Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:

import seaborn as sns
Y_pred = model.predict(X_val)
Y_pred_classes = np.argmax(Y_pred,axis = 1) 
Y_true = np.argmax(Y_val,axis = 1) 
confusion_mtx = confusion_matrix(Y_true, Y_pred_classes) 
f,ax = plt.subplots(figsize=(8, 8))
sns.heatmap(confusion_mtx, annot=True, linewidths=0.01,cmap="Greens",linecolor="gray", fmt= '.1f',ax=ax)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()